In [1]:
import numpy as np
import pandas as pd
import json

In [3]:
X = np.load("../../datasets/Unified/X.npy", allow_pickle=True)
y = np.load("../../datasets/Unified/y.npy", allow_pickle=True)
subjects = np.load("../../datasets/Unified/subjects.npy", allow_pickle=True)
datasets = np.load("../../datasets/Unified/datasets.npy", allow_pickle=True)

with open("../../datasets/Unified/metadata.json", "r") as f:
    metadata = json.load(f)

print("✅ Unified dataset loaded successfully!")

✅ Unified dataset loaded successfully!


In [4]:
print("Checking for NaN values...")

nan_count = np.isnan(X).sum()

print(f"Total NaN values: {nan_count}")

if nan_count == 0:
    print("✅ No NaN values found.")
else:
    print("❌ NaN values detected!")

Checking for NaN values...
Total NaN values: 4239360
❌ NaN values detected!


In [5]:
print("X dtype:", X.dtype)
print("X shape:", X.shape)

# Number of samples containing at least one NaN
samples_with_nan = np.isnan(X).any(axis=(1, 2))

print("\nSamples containing NaNs:", samples_with_nan.sum())
print("Total samples:", len(X))

X dtype: float32
X shape: (55157, 6, 512)

Samples containing NaNs: 1380
Total samples: 55157


In [6]:
nan_dataset_ids = datasets[samples_with_nan]

dataset_ids, counts = np.unique(nan_dataset_ids, return_counts=True)

print(pd.DataFrame({
    "Dataset": dataset_ids,
    "NaN Samples": counts
}))

   Dataset  NaN Samples
0        3         1380


In [7]:
# Keep only valid samples

valid_mask = ~samples_with_nan

X = X[valid_mask]
y = y[valid_mask]
subjects = subjects[valid_mask]
datasets = datasets[valid_mask]

print("Remaining samples:", len(X))

Remaining samples: 53777


In [8]:
print("Remaining NaNs:", np.isnan(X).sum())

if np.isnan(X).sum() == 0:
    print("✅ Dataset is completely clean.")
else:
    print("❌ NaNs still remain.")

Remaining NaNs: 0
✅ Dataset is completely clean.


In [9]:
labels, counts = np.unique(y, return_counts=True)

label_names = {
    0: "Healthy",
    1: "Depressed",
    2: "Stressed"
}

balance_df = pd.DataFrame({
    "Class": [label_names[i] for i in labels],
    "Samples": counts,
    "Percentage": np.round(counts / len(y) * 100, 2)
})

balance_df

,Class,Samples,Percentage
0,Healthy,20227,37.61
1,Depressed,3599,6.69
2,Stressed,29951,55.69


In [10]:
# Flatten each EEG window into one long vector
X_flat = X.reshape(len(X), -1)

_, unique_indices = np.unique(X_flat, axis=0, return_index=True)

duplicates = len(X) - len(unique_indices)

print("Duplicate samples:", duplicates)

Duplicate samples: 882


In [11]:
duplicate_mask = np.ones(len(X), dtype=bool)
duplicate_mask[unique_indices] = False

duplicate_datasets = datasets[duplicate_mask]

dup_ids, dup_counts = np.unique(duplicate_datasets, return_counts=True)

pd.DataFrame({
    "Dataset": dup_ids,
    "Duplicate Samples": dup_counts
})

,Dataset,Duplicate Samples
0,2,882


In [12]:
# Keep only the first occurrence of every EEG window
X = X[unique_indices]
y = y[unique_indices]
subjects = subjects[unique_indices]
datasets = datasets[unique_indices]

print("Remaining samples:", len(X))

Remaining samples: 52895


In [13]:
# Verify duplicates are gone

X_flat = X.reshape(len(X), -1)

_, unique_indices = np.unique(X_flat, axis=0, return_index=True)

print("Remaining duplicate samples:", len(X) - len(unique_indices))

Remaining duplicate samples: 0


In [14]:
import os

# Create folder if it doesn't already exist
os.makedirs("../../datasets/Cleaned", exist_ok=True)

# Save cleaned arrays
np.save("../../datasets/Cleaned/X.npy", X)
np.save("../../datasets/Cleaned/y.npy", y)
np.save("../../datasets/Cleaned/subjects.npy", subjects)
np.save("../../datasets/Cleaned/datasets.npy", datasets)

print("✅ Cleaned dataset saved successfully!")

✅ Cleaned dataset saved successfully!


In [15]:
cleaned_metadata = {
    "summary": {
        "total_samples": len(X),
        "channels": X.shape[1],
        "window_size": X.shape[2],
        "classes": {
            "0": "Healthy",
            "1": "Depressed",
            "2": "Stressed"
        }
    },
    "preprocessing": {
        "nan_samples_removed": 1380,
        "duplicate_samples_removed": 882,
        "final_samples": len(X)
    }
}

with open("../../datasets/Cleaned/metadata.json", "w") as f:
    json.dump(cleaned_metadata, f, indent=4)

print("✅ Metadata saved successfully!")

✅ Metadata saved successfully!
